In [1]:
import time
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import least_squares, minimize

pd.set_option('display.max_columns', None)

#### Load travel-time layers

In [ ]:
path = Path("..").joinpath(
    "data_observation"
)
tt_df = pd.read_csv(path.joinpath("tt_res_data_r3.csv")).sort_values(by=['travelTimeMeta_ID', 'ztop_TT'])

valid = ((tt_df['sigmavp_TT'] > 0) & (tt_df['Qtn_TT'] > 0) &
         (tt_df['Vsi'] > 50) & (tt_df['Vsi'] < 1000))
tt_df = tt_df[valid].reset_index(drop=True)

tt_id_layer = tt_df['travelTimeMeta_ID'].to_numpy()
ztop = tt_df['ztop_TT'].to_numpy()
zbot = tt_df['zbot_TT'].to_numpy()

# Layer assignment below requires non-overlapping layers within each profile
same_profile = tt_id_layer[1:] == tt_id_layer[:-1]
assert not np.any(same_profile & (ztop[1:] < zbot[:-1])), "Overlapping layers within a profile"

#### Load CPT data and assign each point to a layer

In [ ]:
cpt_df = pd.read_csv(path.joinpath("cpt_res_data_r3.csv"))
tt_id_cpt = cpt_df['travelTimeMeta_ID'].to_numpy()
depth_cpt = cpt_df['CPT_depth'].to_numpy()

assert np.all(zbot > ztop), "Zero-thickness layer"
all_ids = np.unique(np.concatenate([tt_id_layer, tt_id_cpt]))
depth_span = max(zbot.max(), depth_cpt.max()) + 1.0
key_layer = np.searchsorted(all_ids, tt_id_layer) * depth_span + ztop
key_cpt = np.searchsorted(all_ids, tt_id_cpt) * depth_span + depth_cpt
cand = np.searchsorted(key_layer, key_cpt, side='right') - 1
cand_above = np.clip(cand - 1, 0, None)
cand = np.clip(cand, 0, None)


def contains(j):
    return (tt_id_layer[j] == tt_id_cpt) & (depth_cpt >= ztop[j]) & (depth_cpt <= zbot[j])

in_cand = contains(cand)
in_above = contains(cand_above) & (cand_above != cand)

# One entry per (CPT point, layer) pair
pair_cpt = np.concatenate([np.flatnonzero(in_cand), np.flatnonzero(in_above)])
pair_layer = np.concatenate([cand[in_cand], cand_above[in_above]])
print(f"{np.sum(in_above)} CPT points lie on a layer boundary and are used in both layers")

8503 CPT points lie on a layer boundary and are used in both layers


#### Filter CPT points, drop layers with no valid points, build model arrays

In [4]:
cpt_ok = ((cpt_df['Qtn_CPT'].to_numpy() > 0) & (cpt_df['sigmavp_CPT'].to_numpy() > 0))
keep = cpt_ok[pair_cpt]
cpt = cpt_df.iloc[pair_cpt[keep]].reset_index(drop=True)
layer = pair_layer[keep]

# Drop layers with no valid CPT points and renumber the remaining layers 0..n-1
N_layer = np.bincount(layer, minlength=len(tt_df))
has_pts = N_layer > 0
print(f"Dropping {np.sum(~has_pts)} layers with no valid CPT points")
layer = (np.cumsum(has_pts) - 1)[layer]
tt_df = tt_df[has_pts].reset_index(drop=True)
N_layer = N_layer[has_pts]

# CPT-point arrays (a boundary point appears once for each layer that uses it)
X1 = cpt['Qtn_CPT'].to_numpy()
X3 = cpt['Ic_CPT'].to_numpy()
lnX2 = np.log(cpt['sigmavp_CPT'].to_numpy())
group_ids, group_index_cpt = np.unique(cpt['group_id'].to_numpy(), return_inverse=True)

# Layer arrays
thickness = (tt_df['zbot_TT'] - tt_df['ztop_TT']).to_numpy()
dz = thickness / N_layer  # each CPT point represents an equal share of its layer
ln_tt = np.log(thickness / tt_df['Vsi'].to_numpy())
profile_ids, profile_index = np.unique(tt_df['travelTimeMeta_ID'].to_numpy(), return_inverse=True)
n_layers_in_profile = np.bincount(profile_index)

group_names = tt_df.groupby('group_id')['group'].first().reindex(group_ids).to_numpy()

print(f"{len(X1)} CPT point-layer pairs, {len(ln_tt)} layers, {len(profile_ids)} profiles, {len(group_ids)} geology groups")
print(f"CPT points per layer: min {N_layer.min()}, median {np.median(N_layer):.0f}, max {N_layer.max()}")
print(f"Single-layer profiles: {np.sum(n_layers_in_profile == 1)}")

Dropping 0 layers with no valid CPT points
442270 CPT point-layer pairs, 12133 layers, 1300 profiles, 11 geology groups
CPT points per layer: min 1, median 41, max 357
Single-layer profiles: 14


#### Model functions

In [ ]:
def ln_vs_cpt(b, a):
    kc = 1 + 5 / (1 + np.exp(-b[1] * X3 + b[2]))
    f_Ic = 0.1 + 0.4 / (1 + np.exp(-b[3] * X3 + b[4]))
    return b[0] * np.log(kc * X1) + f_Ic * lnX2 + a


def layer_residual(b, a):
    tt_hat = dz * np.bincount(layer, weights=np.exp(-ln_vs_cpt(b, a)), minlength=len(dz)) 
    return ln_tt - np.log(tt_hat)


def neg_log_lik(r, tau, d):
    s1 = np.bincount(profile_index, 1.0 / d)
    sr = np.bincount(profile_index, r / d)
    srr = np.bincount(profile_index, r * r / d)
    log_det = np.bincount(profile_index, np.log(d)) + np.log1p(tau**2 * s1)
    quad = srr - tau**2 * sr**2 / (1.0 + tau**2 * s1)
    return 0.5 * np.sum(n_layers_in_profile * np.log(2.0 * np.pi) + log_det + quad)


def random_effects(r, tau, phi_l2l, d):
    s1 = np.bincount(profile_index, 1.0 / d)
    sr = np.bincount(profile_index, r / d)
    eta_p = (tau**2 * sr / (1.0 + tau**2 * s1))[profile_index]
    eta_l = phi_l2l**2 / d * (r - eta_p)
    return eta_p, eta_l, r - eta_p - eta_l


def fit(objective, x0, n_sigma, name):
    bounds = [(None, None)] * (len(x0) - n_sigma) + [(1e-3, 2.0)] * n_sigma
    start_time = time.time()
    res = minimize(objective, x0, bounds=bounds, method='L-BFGS-B',
                   options={'maxiter': 5000, 'maxfun': 200000})
    print(f"{name}: {time.time() - start_time:.1f} s, success={res.success}, {res.message}")
    print(f"  -logL = {res.fun:.2f}, AIC = {2 * res.fun + 2 * len(x0):.2f}")
    return res


summary = {}

#### Model 1: Baseline model, no random effects

In [6]:
def baseline_residual(x):
    return layer_residual(x[:5], x[5])

x0 = [0.25, 5, 10, 5, 5, 4]
start_time = time.time()
result1 = least_squares(baseline_residual, x0)
print(f"Model 1: {time.time() - start_time:.1f} s, success={result1.success}, {result1.message}")

b_m1 = result1.x
sigma_m1 = np.sqrt(np.mean(result1.fun**2))  # MLE of sigma
nll_m1 = neg_log_lik(result1.fun, 0.0, np.full(len(ln_tt), sigma_m1**2))
summary['1: baseline'] = dict(zip(['b1', 'b2', 'b3', 'b4', 'b5', 'a'], b_m1),
                               sigma_total=sigma_m1, nll=nll_m1, k=len(x0) + 1)
print("b1..b5, a =", np.round(b_m1, 4))
print(f"sigma = {sigma_m1:.4f}, -logL = {nll_m1:.2f}")

Model 1: 2.3 s, success=True, `ftol` termination condition is satisfied.
b1..b5, a = [ 0.2667  9.6062 21.231   1.476   3.9416  4.1348]
sigma = 0.2590, -logL = 826.67


#### Model 2: Profile-to-profile

In [7]:
def p2p_model(x):
    tau, phi = x[6], x[7]
    return neg_log_lik(layer_residual(x[:5], x[5]), tau, np.full(len(ln_tt), phi**2))

result2 = fit(p2p_model, list(b_m1) + [0.15, 0.2], 2, "Model 2")
b1, b2, b3, b4, b5, a, phi_p2p, phi_ep = result2.x
summary['2: p2p'] = dict(b1=b1, b2=b2, b3=b3, b4=b4, b5=b5, a=a, phi_p2p=phi_p2p, phi_ep=phi_ep,
                         sigma_total=np.hypot(phi_p2p, phi_ep), nll=result2.fun, k=len(result2.x))
pd.Series(summary['2: p2p'])

Model 2: 24.0 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -140.81, AIC = -265.62


b1               0.235050
b2               5.690069
b3              12.776113
b4               1.276103
b5               3.305903
a                4.281358
phi_p2p          0.144517
phi_ep           0.220428
sigma_total      0.263579
nll           -140.810700
k                8.000000
dtype: float64

#### Model 3: Profile-to-profile + layer-to-layer + within-layer

In [8]:
def l2l_model(x):
    tau, phi_l2l, phi_wl = x[6], x[7], x[8]
    d = phi_l2l**2 + phi_wl**2 / N_layer
    return neg_log_lik(layer_residual(x[:5], x[5]), tau, d)

result3 = fit(l2l_model, list(result2.x[:7]) + [0.15, 0.3], 3, "Model 3")
b1, b2, b3, b4, b5, a, phi_p2p, phi_l2l, phi_wl = result3.x
summary['3: p2p + l2l'] = dict(b1=b1, b2=b2, b3=b3, b4=b4, b5=b5, a=a, phi_p2p=phi_p2p,
                               phi_l2l=phi_l2l, phi_wl=phi_wl,
                               sigma_total=np.sqrt(phi_p2p**2 + phi_l2l**2 + phi_wl**2 / np.median(N_layer)),
                               nll=result3.fun, k=len(result3.x))
pd.Series(summary['3: p2p + l2l'])

Model 3: 35.5 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -238.57, AIC = -459.13


b1               0.236220
b2               5.942227
b3              13.275427
b4               1.496138
b5               3.821757
a                4.275154
phi_p2p          0.145239
phi_l2l          0.196932
phi_wl           0.463937
sigma_total      0.255198
nll           -238.567066
k                9.000000
dtype: float64

In [9]:
print(b3/b2)
print(b5/b4)

2.2340828107831765
2.5544143267126747


#### Model 4: Profile-to-profile + layer-to-layer + within-layer, with geology-specific intercepts

In [10]:
n_groups = len(group_ids)

def geo_model(x):
    a = x[5:5 + n_groups][group_index_cpt]
    tau, phi_l2l, phi_wl = x[5 + n_groups:]
    d = phi_l2l**2 + phi_wl**2 / N_layer
    return neg_log_lik(layer_residual(x[:5], a), tau, d)

x0 = list(result3.x[:5]) + [result3.x[5]] * n_groups + list(result3.x[6:])
result4 = fit(geo_model, x0, 3, "Model 4")
b1, b2, b3, b4, b5 = result4.x[:5]
a_geo = result4.x[5:5 + n_groups]
phi_p2p, phi_l2l, phi_wl = result4.x[5 + n_groups:]
summary['4: geology'] = dict(b1=b1, b2=b2, b3=b3, b4=b4, b5=b5, phi_p2p=phi_p2p,
                             phi_l2l=phi_l2l, phi_wl=phi_wl,
                             sigma_total=np.sqrt(phi_p2p**2 + phi_l2l**2 + phi_wl**2 / np.median(N_layer)),
                             nll=result4.fun, k=len(result4.x))

pd.DataFrame({'group_id': group_ids, 'group': group_names, 'a': a_geo,
              'n_profiles': [len(np.unique(cpt['travelTimeMeta_ID'][group_index_cpt == i]))
                             for i in range(n_groups)]})

Model 4: 193.4 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -447.57, AIC = -857.14


,group_id,group,a,n_profiles
0,1,Beach_Dune,4.402626,152
1,2,Floodplain,4.255830,352
2,3,Glacial,4.195240,76
3,4,Lagoonal,4.195468,30
4,5,Valley_Train,4.238807,72
5,6,af/Qi,4.316504,92
6,7,Qal1,4.350243,89
7,8,Qal2,4.411610,278
8,9,Qal3,4.383580,64
9,10,Qoa,4.515429,74


#### Model comparison

`sigma_total` is the standard deviation of a layer's residual, using the median $N_l$ for the within-layer term in Models 3 and 4.

In [11]:
comparison = pd.DataFrame(summary).T
comparison['AIC'] = 2 * comparison['nll'] + 2 * comparison['k']
comparison

,b1,b2,b3,b4,b5,a,sigma_total,nll,k,phi_p2p,phi_ep,phi_l2l,phi_wl,AIC
1: baseline,0.266708,9.606246,21.230985,1.476008,3.941569,4.134776,0.259032,826.673569,7.0,NaN,NaN,NaN,NaN,1667.347137
2: p2p,0.235050,5.690069,12.776113,1.276103,3.305903,4.281358,0.263579,-140.810700,8.0,0.144517,0.220428,NaN,NaN,-265.621400
3: p2p + l2l,0.236220,5.942227,13.275427,1.496138,3.821757,4.275154,0.255198,-238.567066,9.0,0.145239,NaN,0.196932,0.463937,-459.134132
4: geology,0.224117,5.482742,12.320768,1.278293,3.385425,NaN,0.240965,-447.568244,19.0,0.118555,NaN,0.197248,0.457373,-857.136488


#### Random effects (Model 4 BLUPs)

In [12]:
r4 = layer_residual(result4.x[:5], a_geo[group_index_cpt])
d4 = phi_l2l**2 + phi_wl**2 / N_layer
eta_p, eta_l, eps = random_effects(r4, phi_p2p, phi_l2l, d4)

resid_df = tt_df[['travelTimeMeta_ID', 'Ic_TT','ztop_TT', 'zbot_TT', 'Vsi', 'group', 'group_id']].assign(
    N_cpt=N_layer, residual=r4, eta_p=eta_p, eta_l=eta_l, eps=eps)
first_layer = np.unique(profile_index, return_index=True)[1]
print("std of eta_p (one per profile):", np.std(eta_p[first_layer]).round(4))
print("std of eta_l:", np.std(eta_l).round(4), " std of eps:", np.std(eps).round(4))
resid_df.head()

std of eta_p (one per profile): 0.0997
std of eta_l: 0.1729  std of eps: 0.0523


,travelTimeMeta_ID,Ic_TT,ztop_TT,zbot_TT,Vsi,group,group_id,N_cpt,residual,eta_p,eta_l,eps
0,1.0,1.841227,1.75,3.75,188.225684,Beach_Dune,1,41,0.182169,0.093605,0.078296,0.010268
1,1.0,1.640247,3.75,5.75,259.666942,Beach_Dune,1,41,0.060636,0.093605,-0.029147,-0.003822
2,1.0,1.586969,5.75,7.75,256.477847,Beach_Dune,1,41,0.216475,0.093605,0.108624,0.014245
3,1.0,1.620394,7.75,9.55,270.007966,Beach_Dune,1,37,0.209668,0.093605,0.101337,0.014726
4,2.0,1.698607,1.75,3.75,214.500849,Beach_Dune,1,41,0.116910,0.098281,0.016469,0.002160


#### Model 4: stratified 5-fold cross-validation

In [ ]:
from contextlib import contextmanager
from sklearn.model_selection import StratifiedKFold

data_names = ['X1', 'X3', 'lnX2', 'group_index_cpt', 'layer', 'dz', 'ln_tt', 'N_layer',
              'profile_index', 'n_layers_in_profile']
full_data = {name: globals()[name] for name in data_names}


def subset(in_set):
    d = full_data
    keep_layer = in_set[d['profile_index']]
    keep_pt = keep_layer[d['layer']]
    new_profile = np.unique(d['profile_index'][keep_layer], return_inverse=True)[1]
    return dict(X1=d['X1'][keep_pt], X3=d['X3'][keep_pt], lnX2=d['lnX2'][keep_pt],
                group_index_cpt=d['group_index_cpt'][keep_pt],
                layer=(np.cumsum(keep_layer) - 1)[d['layer'][keep_pt]],
                dz=d['dz'][keep_layer], ln_tt=d['ln_tt'][keep_layer], N_layer=d['N_layer'][keep_layer],
                profile_index=new_profile, n_layers_in_profile=np.bincount(new_profile))


@contextmanager
def use_data(data):
    try:
        globals().update(data)
        yield
    finally:
        globals().update(full_data)

profile_group = tt_df.groupby('travelTimeMeta_ID')['group_id'].first().reindex(profile_ids).to_numpy()
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = []
for train_p, test_p in skf.split(profile_ids, profile_group):
    in_test = np.zeros(len(profile_ids), dtype=bool)
    in_test[test_p] = True
    folds.append(in_test)

nll_parts = []
for in_test in folds:
    with use_data(subset(in_test)):
        nll_parts.append(geo_model(result4.x))
assert np.isclose(sum(nll_parts), geo_model(result4.x)), "Fold subsets do not partition the data"

cv_rows, cv_params = [], []
oof_resid = np.full(len(ln_tt), np.nan)
for k, in_test in enumerate(folds):
    with use_data(subset(~in_test)):
        res = fit(geo_model, result4.x, 3, f"Fold {k + 1}")
        n_train = len(ln_tt)
        r_train = layer_residual(res.x[:5], res.x[5:5 + n_groups][group_index_cpt])
        N_med = np.median(N_layer)
    with use_data(subset(in_test)):
        n_test = len(ln_tt)
        nll_test = geo_model(res.x)
        r_test = layer_residual(res.x[:5], res.x[5:5 + n_groups][group_index_cpt])
    oof_resid[in_test[profile_index]] = r_test

    tau, phi_l2l_k, phi_wl_k = res.x[5 + n_groups:]
    rmse_train, rmse_test = np.sqrt(np.mean(r_train**2)), np.sqrt(np.mean(r_test**2))
    cv_rows.append(dict(fold=k + 1, test_profiles=in_test.sum(), test_layers=n_test,
                        nll_per_layer_train=res.fun / n_train, nll_per_layer_test=nll_test / n_test,
                        nll_gap=nll_test / n_test - res.fun / n_train,
                        rmse_train=rmse_train, rmse_test=rmse_test, overfit_ratio=rmse_test / rmse_train,
                        bias_test=r_test.mean(),
                        sigma_total=np.sqrt(tau**2 + phi_l2l_k**2 + phi_wl_k**2 / N_med)))
    cv_params.append(dict(zip(['b1', 'b2', 'b3', 'b4', 'b5'], res.x[:5]),
                          **dict(zip(group_names, res.x[5:5 + n_groups])),
                          phi_p2p=tau, phi_l2l=phi_l2l_k, phi_wl=phi_wl_k))

cv_results = pd.DataFrame(cv_rows).set_index('fold')
cv_results.loc['mean'] = cv_results.mean()
cv_results.loc['std'] = cv_results.iloc[:5].std()
cv_results

Fold 1: 159.5 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -427.52, AIC = -817.04
Fold 2: 127.8 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -272.72, AIC = -507.44
Fold 3: 136.1 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -311.61, AIC = -585.21
Fold 4: 106.1 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -328.26, AIC = -618.52
Fold 5: 126.1 s, success=True, CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  -logL = -464.44, AIC = -890.88


,test_profiles,test_layers,nll_per_layer_train,nll_per_layer_test,nll_gap,rmse_train,rmse_test,overfit_ratio,bias_test,sigma_total
fold,,,,,,,,,,
1,260.0,2523.000000,-0.044487,-0.006429,0.038058,0.248164,0.259944,1.047469,0.016860,0.239873
2,260.0,2270.000000,-0.027651,-0.075583,-0.047932,0.251705,0.240391,0.955050,0.003621,0.243889
3,260.0,2346.000000,-0.031839,-0.056378,-0.024539,0.250692,0.247744,0.988239,0.021232,0.241202
4,260.0,2468.000000,-0.033964,-0.047404,-0.013440,0.252206,0.243005,0.963519,0.023464,0.242850
5,260.0,2526.000000,-0.048344,0.008563,0.056907,0.246890,0.261380,1.058690,0.017440,0.236568
mean,260.0,2426.600000,-0.037257,-0.035446,0.001811,0.249931,0.250493,1.002593,0.016523,0.240876
std,0.0,113.915758,0.008770,0.035252,0.044019,0.002306,0.009663,0.047838,0.007710,0.002857


#### Cross-validation: parameter stability across folds

In [14]:
cv_param_df = pd.DataFrame(cv_params, index=pd.Index(range(1, 6), name='fold'))
cv_param_df.loc['full data'] = result4.x
cv_param_df.loc['fold std'] = cv_param_df.iloc[:5].std()
cv_param_df.T

fold,1,2,3,4,5,full data,fold std
b1,0.224431,0.223877,0.223882,0.223569,0.224429,0.224117,0.000380
b2,5.064568,5.650542,5.650771,5.384579,5.642704,5.482742,0.258069
b3,11.450274,12.686627,12.689028,12.068572,12.664477,12.320768,0.549673
b4,1.234952,1.216618,1.397606,1.185219,1.352939,1.278293,0.092398
b5,3.272862,3.216792,3.675939,3.152468,3.600734,3.385425,0.237760
Beach_Dune,4.392584,4.405821,4.401774,4.414813,4.400067,4.402626,0.008156
Floodplain,4.255781,4.256371,4.260696,4.256376,4.251678,4.255830,0.003197
Glacial,4.197968,4.197099,4.199758,4.194926,4.187646,4.195240,0.004711
Lagoonal,4.202518,4.183015,4.193689,4.194181,4.205667,4.195468,0.008846
Valley_Train,4.225905,4.240711,4.244504,4.238529,4.245764,4.238807,0.007916


#### Cross-validation: out-of-fold residuals by geology group

In [15]:
resid_df.assign(oof_residual=oof_resid).groupby('group')['oof_residual'].agg(
    n_layers='count', mean='mean', std='std', rmse=lambda r: np.sqrt(np.mean(r**2)))

,n_layers,mean,std,rmse
group,,,,
Beach_Dune,1229,-0.006396,0.316327,0.316263
Floodplain,3009,0.003042,0.184327,0.184322
Glacial,588,0.001584,0.201661,0.201496
Lagoonal,278,0.006841,0.281401,0.280978
Qal1,1040,0.024760,0.230769,0.231983
Qal2,2314,0.026586,0.274505,0.275731
Qal3,438,0.025142,0.275722,0.276553
Qoa,1520,0.060742,0.277994,0.284463
Qs,140,0.001425,0.286469,0.285448
